# EDA · Kiniela Predictor

**Responsable:** Anna · **Issue:** #4

Análisis exploratorio de la Premier League 2023-24 para decidir cómo modelar las 6 clases de `target`. Toda la lógica (carga, `target` y features) vive en `src/kiniela/`; aquí solo se llama y se mira.

**Datos:** `data/processed/premier_league_2023_24.csv`. No está versionado: se genera con `download_premier_league_data("2324").to_csv("data/processed/premier_league_2023_24.csv", index=False)` (ver `docs/tarea-preparacion-dataset.md`). Las fechas pueden venir como `dd/mm/aaaa` o en formato ISO.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# The package lives in src/: make it importable when the notebook runs from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from kiniela.config import CLASS_NAMES
from kiniela.data.dataset import load_matches
from kiniela.data.target import build_target
# _sorted_by_date is the date parsing of the features, also used by models/final.py
from kiniela.features.engineering import FEATURE_COLUMNS, _sorted_by_date, build_features

sns.set_theme(style="whitegrid")

## 1. Dataset

In [ ]:
matches = load_matches(ROOT / "data" / "processed" / "premier_league_2023_24.csv")
matches = _sorted_by_date(matches)  # dates may be dd/mm/yyyy (football-data) or ISO (regenerated CSV)

print("Matches:", len(matches), "| columns:", matches.shape[1])
print("From", matches["Date"].min().date(), "to", matches["Date"].max().date())
print("Teams:", matches["HomeTeam"].nunique())
print("Home matches per team:", matches["HomeTeam"].value_counts().unique())
print("Missing values in the main columns:")
print(matches[["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]].isna().sum().to_string())

## 2. Variable objetivo

Regla de `src/kiniela/data/target.py`: la clase 5 (4 o más goles en total) tiene prioridad; si no, una diferencia de 2 o más goles es victoria clara (0 o 4) y el resto son resultados ajustados (1, 2 o 3).

In [ ]:
matches["target"] = [build_target(home, away) for home, away in zip(matches["FTHG"], matches["FTAG"])]

class_counts = matches["target"].value_counts().sort_index()
class_table = pd.DataFrame(
    {
        "class": [CLASS_NAMES[c] for c in class_counts.index],
        "matches": class_counts.values,
        "percent": (class_counts / len(matches) * 100).round(1).values,
    },
    index=class_counts.index,
)
print(class_table.to_string())
print("Largest / smallest class:", round(class_counts.max() / class_counts.min(), 1))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(x=class_table.index, y=class_table["percent"], color="steelblue", ax=ax)
ax.set_xlabel("Class")
ax.set_ylabel("% of matches")
ax.set_title("Class distribution · Premier League 2023-24")
for position, value in enumerate(class_table["percent"]):
    ax.text(position, value + 0.5, f"{value}%", ha="center")
plt.show()

## 3. Goles

La clase 5 depende solo del total de goles: conviene ver cuántos partidos quedan justo por debajo y por encima de 4.

In [ ]:
total_goals = matches["FTHG"] + matches["FTAG"]
print("Goals per match:", round(total_goals.mean(), 2),
      "| home:", round(matches["FTHG"].mean(), 2), "| away:", round(matches["FTAG"].mean(), 2))
print("Full-time result (%):", (matches["FTR"].value_counts(normalize=True) * 100).round(1).to_dict())
print("Matches with 3 goals:", (total_goals == 3).sum(), "| with 4:", (total_goals == 4).sum())

fig, ax = plt.subplots(figsize=(9, 4))
sns.countplot(x=total_goals, color="steelblue", ax=ax)
ax.axvline(3.5, color="crimson", linestyle="--", label="class 5 threshold (4+ goals)")
ax.set_xlabel("Total goals in the match")
ax.set_ylabel("Matches")
ax.legend()
plt.show()

## 4. Features históricas

`build_features` calcula cada feature solo con partidos anteriores de la misma temporada, así que las primeras jornadas no tienen historial. Para el análisis se usan los partidos con todas las features.

In [ ]:
features = build_features(matches)
features["target"] = [build_target(home, away) for home, away in zip(features["FTHG"], features["FTAG"])]
complete = features.dropna(subset=list(FEATURE_COLUMNS))

print("Matches with every feature:", len(complete), "of", len(features),
      f"({len(features) - len(complete)} without history are dropped)")
print(complete[list(FEATURE_COLUMNS)].describe().round(2).to_string())

### Media de cada feature por clase

`target` es una variable nominal (la clase 5 no está «entre» la 4 y nada), así que en lugar de una correlación con `target` se compara la media de cada feature en cada clase.

In [ ]:
by_class = complete.groupby("target")[list(FEATURE_COLUMNS)].mean().round(2)
print(by_class.to_string())

### Correlación entre features

In [ ]:
correlation = complete[list(FEATURE_COLUMNS)].corr(method="spearman")

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, ax=ax)
ax.set_title("Spearman correlation between features")
plt.show()

# Upper triangle only: each pair once, without the diagonal.
upper = np.triu(np.ones(correlation.shape, dtype=bool), k=1)
pairs = correlation.where(upper).stack()
print("Pairs with |rho| > 0.7:")
print(pairs[pairs.abs() > 0.7].sort_values(ascending=False).round(2).to_string())

## 5. Conclusiones para el modelado

Cifras de la Premier League 2023-24 (380 partidos) obtenidas al ejecutar este notebook.

1. **Clases muy desbalanceadas.** La clase 5 (4+ goles) tiene el 43,7 % de los partidos y la clase 4 (victoria visitante clara) solo el 6,8 %: la mayor es 6,4 veces la menor. La accuracy engaña: un modelo que siempre dice «clase 5» acierta casi la mitad. Hay que validar con `StratifiedKFold` y comparar modelos con **F1 macro**, y ponderar las clases al entrenar.
2. **La clase 5 depende solo del total de goles y la frontera es estrecha.** Hay 80 partidos con 3 goles y otros 80 con 4: un gol decide si un partido es clase 5 o una de las otras. Es la clase más frecuente, pero no la más fácil.
3. **Las features separan poco las clases.** Las medias por clase se parecen mucho. La señal más clara está entre victoria local y victoria visitante: `match_diff` vale 0,44 de media en la clase 0 y -0,02 en la clase 4. En la clase 5, `match_goals_expected` (3,28) apenas se distingue de la media general (3,22). Hay que esperar un F1 macro bajo con estas features.
4. **Features muy correlacionadas entre sí.** La forma y la diferencia de goles del mismo equipo van casi juntas (ρ = 0,92 en casa y 0,89 fuera), y también los goles y la diferencia de goles (ρ = 0,87 en casa). Los modelos de árboles lo toleran; un modelo lineal necesita regularización. Eliminar alguna feature es una decisión del equipo, no de este análisis.
5. **Las primeras jornadas no tienen historial.** 21 de los 380 partidos no tienen todas las features y se descartan al entrenar: quedan 359 partidos.
6. **Hay pocos datos.** Con una sola temporada, cada clase pequeña tiene unas decenas de partidos (26 en la clase 4). Por eso el modelo final (#66) se entrena con 9 temporadas, de 2015-16 a 2023-24 (245 partidos útiles de la clase 4): con una temporada el mejor modelo no superaba al de prueba (F1 macro 0,095 frente a 0,102) y con 9 llega a 0,172 frente a 0,080.

Ninguna conclusión usa el resultado de un partido como feature de ese mismo partido: `FTHG`, `FTAG` y el total de goles solo sirven aquí para describir `target`.